In [10]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import BoydSolver
from signalClass import *
import time

In [11]:
np.random.seed(24102000)
n = 128

construct blur matrix

In [12]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [13]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 15)
RndSignal.generate_GG_realization(0, sigma, 2)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [14]:
mu = 0.5
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [15]:
#begin solver construction
np.random.seed(24102001)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = BoydSolver.BoydSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [16]:
iters = 2500

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [17]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 2500
2 / 2500
3 / 2500
4 / 2500
5 / 2500
6 / 2500
7 / 2500
8 / 2500
9 / 2500
10 / 2500
11 / 2500
12 / 2500
13 / 2500
14 / 2500
15 / 2500
16 / 2500
17 / 2500
18 / 2500
19 / 2500
20 / 2500
21 / 2500
22 / 2500
23 / 2500
24 / 2500
25 / 2500
26 / 2500
27 / 2500
28 / 2500
29 / 2500
30 / 2500
31 / 2500
32 / 2500
33 / 2500
34 / 2500
35 / 2500
36 / 2500
37 / 2500
38 / 2500
39 / 2500
40 / 2500
41 / 2500
42 / 2500
43 / 2500
44 / 2500
45 / 2500
46 / 2500
47 / 2500
48 / 2500
49 / 2500
50 / 2500
51 / 2500
52 / 2500
53 / 2500
54 / 2500
55 / 2500
56 / 2500
57 / 2500
58 / 2500
59 / 2500
60 / 2500
61 / 2500
62 / 2500
63 / 2500
64 / 2500
65 / 2500
66 / 2500
67 / 2500
68 / 2500
69 / 2500
70 / 2500
71 / 2500
72 / 2500
73 / 2500
74 / 2500
75 / 2500
76 / 2500
77 / 2500
78 / 2500
79 / 2500
80 / 2500
81 / 2500
82 / 2500
83 / 2500
84 / 2500
85 / 2500
86 / 2500
87 / 2500
88 / 2500
89 / 2500
90 / 2500
91 / 2500
92 / 2500
93 / 2500
94 / 2500
95 / 2500
96 / 2500
97 / 2500
98 / 2500
99 / 2500
100 / 2500
101 / 25

In [18]:
np.savez_compressed(
    "./BoydADMMTVL1-Gauss.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)